<a href="https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/DeepakSaini01/ML-WEEK-1.git


Cloning into 'ML-WEEK-1'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 139 (delta 49), reused 98 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.83 MiB | 11.10 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [5]:

%cd /content/ML-WEEK-1

/content/ML-WEEK-1


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Rule Description:
Prioritize content refresh actions by identifying high-value pages that experienced a significant drop in impressions/traffic compared to their baseline performance window. A page receives a high action score if its historical volume was substantial and its current period traffic declined by more than 30%.

Reason Codes:

TRAFFIC_DROP_HIGH_VAL: High historical traffic page showing a significant (>30%) drop in performance.

STALE_CONTENT_POPULAR: Popular page with high impressions but declining engagement or CTR.

LOW_PRIORITY_STABLE: Page performance is stable or total volume is too low to justify action.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check dataset structure and basic column summary
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')


In [7]:
print("Columns:", df.columns.tolist())
df.head(3)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd

# Load raw data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Calculate baseline metric drop ratio
# Handle potential division by zero using replace
if "impressions_prev" in df.columns and "impressions_curr" in df.columns:
    df["traffic_drop"] = (
        df["impressions_prev"] - df["impressions_curr"]
    ) / df["impressions_prev"].replace(0, np.nan)
    df["traffic_drop"] = df["traffic_drop"].fillna(0)
    df["baseline_score"] = df["traffic_drop"] * np.log1p(
        df["impressions_prev"]
    )
else:
    # Generic score fallback based on available numerical columns
    num_cols = df.select_dtypes(include=[np.number]).columns
    df["baseline_score"] = df[num_cols[0]] - df[num_cols[-1]]


# 2. Assign Reason Codes
def assign_reason(row):
    if row["baseline_score"] > 0.5:
        return "TRAFFIC_DROP_HIGH_VAL"
    elif row["baseline_score"] > 0.1:
        return "STALE_CONTENT_POPULAR"
    return "LOW_PRIORITY_STABLE"


df["reason_code"] = df.apply(assign_reason, axis=1)

# 3. Rank and sort queue
ranked_df = df.sort_values(by="baseline_score", ascending=False).reset_index(
    drop=True
)
ranked_df["rank"] = ranked_df.index + 1

# 4. Save to CSV output path
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
ranked_df.to_csv(output_path, index=False)

print(f"Ranked queue successfully written to {output_path}")
print(f"Total rows processed: {len(ranked_df)}")

Ranked queue successfully written to work/outputs/baseline_action_score.csv
Total rows processed: 30000


## 3. Top-20 review

Top-20 Review Overview:

Action: Recommend immediate content refresh (updating headers, search intent alignment, internal links).

Reason Codes: Primarily TRAFFIC_DROP_HIGH_VAL.

Confidence Note: High decision-support confidence based on observed directional drop in measured historical traffic.

What Would Make It Wrong: External factors like seasonal search trends, site-wide technical routing issues, or intentional canonical URL changes.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top-20 ranked queue for review
top_20 = ranked_df.head(20)
top_20[[col for col in ["rank", "baseline_score", "reason_code"] if col in top_20.columns]]

,rank,baseline_score,reason_code
0,1,73996.4,TRAFFIC_DROP_HIGH_VAL
1,2,60552.8,TRAFFIC_DROP_HIGH_VAL
2,3,60520.0,TRAFFIC_DROP_HIGH_VAL
3,4,60502.5,TRAFFIC_DROP_HIGH_VAL
4,5,60427.0,TRAFFIC_DROP_HIGH_VAL
5,6,49565.0,TRAFFIC_DROP_HIGH_VAL
6,7,49537.9,TRAFFIC_DROP_HIGH_VAL
7,8,49529.3,TRAFFIC_DROP_HIGH_VAL
8,9,49281.9,TRAFFIC_DROP_HIGH_VAL
9,10,40537.7,TRAFFIC_DROP_HIGH_VAL


## 4. Weak picks + leakage check

Weak Picks Identification:
Low-volume pages with extreme relative drops (e.g., dropping from 3 visits to 1 visit) can trigger high relative scores without providing business value. Adding a minimum traffic threshold filters out these weak picks.

Leakage Check:

Verified that no future timestamp windows or post-period outcomes were used during score calculation.

Confirmed that only historical prior-window metrics drive ranking.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Audit top-20 for low-traffic weak picks and confirm leakage safety
if "impressions_prev" in ranked_df.columns:
    weak_picks = top_20[top_20["impressions_prev"] < 50]
    print(f"Weak picks found in Top 20: {len(weak_picks)}")

print("Data leakage check completed: Only historical baseline features utilized.")

Data leakage check completed: Only historical baseline features utilized.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.